In [1]:
# Cell 1 — Konfiguration
import json, time, os
from pathlib import Path
from datetime import datetime, date, timezone
import requests

# Notebook ligger i script/ — projektroten är en nivå upp
NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "script" else NOTEBOOK_DIR
RAW_DIR = PROJECT_ROOT / "raw" / "linear"
RAW_DIR.mkdir(parents=True, exist_ok=True)

CREDENTIALS_PATH = PROJECT_ROOT / "credentials" / "Linear_API-key.txt"
LINEAR_API_URL   = "https://api.linear.app/graphql"

# Backfill-intervall
BACKFILL_FROM = date(2025, 5, 1)
BACKFILL_TO   = datetime.now(timezone.utc).date()

# Pagination / retry — samma värden som linear_snapshot_claude.py
PAGE_SIZE     = 100
MAX_RETRIES   = 5
RETRY_BACKOFF = 2
USER_AGENT    = "puttaren-agent/1.0"

print(f"PROJECT_ROOT:   {PROJECT_ROOT}")
print(f"RAW_DIR:        {RAW_DIR}")
print(f"Backfill range: {BACKFILL_FROM} → {BACKFILL_TO}")

PROJECT_ROOT:   C:\Users\michael.brostrom\Documents\GitHub\statistik-freshdesk-linear
RAW_DIR:        C:\Users\michael.brostrom\Documents\GitHub\statistik-freshdesk-linear\raw\linear
Backfill range: 2025-05-01 → 2026-06-04


In [2]:
# Cell 2 — API-nyckel
def read_api_key() -> str:
    key = os.environ.get("LINEAR_API_KEY", "").strip()
    if key:
        print("Använder LINEAR_API_KEY från miljövariabel.")
        return key
    print(f"Läser API-nyckel från: {CREDENTIALS_PATH}")
    if not CREDENTIALS_PATH.exists():
        raise FileNotFoundError(f"API-nyckel saknas: {CREDENTIALS_PATH}")
    key = CREDENTIALS_PATH.read_text(encoding="utf-8").strip()
    if not key:
        raise ValueError("API-nyckeln är tom.")
    if key.lower().startswith("bearer "):
        key = key.split(None, 1)[1].strip()
    return key

In [3]:
# Cell 3 — GraphQL query och fetch
# Query matchar linear_snapshot_claude.py exakt
GRAPHQL_QUERY = """
query($first: Int!, $after: String) {
  issues(first: $first, after: $after, includeArchived: true) {
    nodes {
      id
      number
      identifier
      title
      description
      createdAt
      updatedAt
      archivedAt
      completedAt
      canceledAt
      startedAt
      dueDate
      priority
      estimate
      trashed
      state        { id name type }
      assignee     { id name email }
      project      { id name }
      team         { id name }
      labels       { nodes { name } }
      parent       { id identifier }
      cycle        { id name startsAt endsAt }
    }
    pageInfo { hasNextPage endCursor }
  }
}
"""


def fetch_all_issues(api_key: str) -> list:
    headers = {
        "Authorization": api_key,
        "Content-Type": "application/json",
        "User-Agent": USER_AGENT,
    }
    issues = []
    cursor = None
    page = 0

    while True:
        page += 1
        payload = {"query": GRAPHQL_QUERY, "variables": {"first": PAGE_SIZE, "after": cursor}}
        attempt = 0

        while True:
            attempt += 1
            try:
                r = requests.post(LINEAR_API_URL, headers=headers, json=payload, timeout=60)
            except requests.RequestException as ex:
                if attempt >= MAX_RETRIES:
                    raise
                wait = RETRY_BACKOFF ** attempt
                print(f"  Request error, retry {attempt}/{MAX_RETRIES} om {wait}s: {ex}")
                time.sleep(wait)
                continue

            if r.status_code == 401:
                raise RuntimeError("Autentisering misslyckades (401) — kontrollera API-nyckeln.")

            if r.status_code == 429:
                if attempt >= MAX_RETRIES:
                    r.raise_for_status()
                wait = int(r.headers.get("Retry-After", RETRY_BACKOFF ** attempt))
                print(f"  Rate limited (429), väntar {wait}s (Retry-After)...")
                time.sleep(wait)
                continue

            if r.status_code >= 500:
                if attempt >= MAX_RETRIES:
                    r.raise_for_status()
                wait = RETRY_BACKOFF ** attempt
                print(f"  Serverfel {r.status_code}, retry {attempt}/{MAX_RETRIES} om {wait}s...")
                time.sleep(wait)
                continue

            break

        try:
            data = r.json()
        except ValueError:
            raise RuntimeError(f"Ogiltigt JSON-svar (sida {page}): {r.text[:500]}")

        if "errors" in data:
            raise RuntimeError(f"GraphQL error (sida {page}): {data['errors']}")

        issues_data = data.get("data") or {}
        if not issues_data:
            raise RuntimeError(f"Saknar 'data' i API-svar (sida {page}): {str(data)[:300]}")

        block = issues_data.get("issues", {}).get("nodes", [])
        print(f"  Sida {page}: {len(block)} noder")
        issues.extend(block)

        page_info = issues_data.get("issues", {}).get("pageInfo", {})
        if page_info.get("hasNextPage"):
            cursor = page_info.get("endCursor")
            if not cursor:
                print(f"  Varning: hasNextPage=True men endCursor saknas (sida {page}) — avslutar.")
                break
        else:
            break

    return issues

In [4]:
# Cell 4 — Kör backfill och spara
api_key = read_api_key()

print("Hämtar alla issues från Linear...")
all_issues = fetch_all_issues(api_key)
print(f"Totalt hämtade: {len(all_issues)} noder")

# Filtrera på createdAt >= BACKFILL_FROM (klientsida — enkel och transparent)
def created_on_or_after(node: dict, from_date: date) -> bool:
    raw = node.get("createdAt", "")
    if not raw:
        return False
    return date.fromisoformat(raw[:10]) >= from_date

filtered = [n for n in all_issues if created_on_or_after(n, BACKFILL_FROM)]
excluded = len(all_issues) - len(filtered)
print(f"Efter filter (createdAt >= {BACKFILL_FROM}): {len(filtered)} noder ({excluded} exkluderade)")

# Spara
out_name = f"linear_backfill_{BACKFILL_FROM.strftime('%Y%m%d')}_{BACKFILL_TO.strftime('%Y%m%d')}.json"
out_path = RAW_DIR / out_name
with out_path.open("w", encoding="utf-8") as f:
    json.dump(filtered, f, ensure_ascii=False, indent=2)

print(f"\nSparade: {out_path}")
print(f"Filstorlek: {out_path.stat().st_size / 1024:.0f} KB")

Läser API-nyckel från: C:\Users\michael.brostrom\Documents\GitHub\statistik-freshdesk-linear\credentials\Linear_API-key.txt
Hämtar alla issues från Linear...
  Sida 1: 100 noder
  Sida 2: 100 noder
  Sida 3: 100 noder
  Sida 4: 100 noder
  Sida 5: 100 noder
  Sida 6: 100 noder
  Sida 7: 100 noder
  Sida 8: 100 noder
  Sida 9: 100 noder
  Sida 10: 100 noder
  Sida 11: 89 noder
Totalt hämtade: 1089 noder
Efter filter (createdAt >= 2025-05-01): 1089 noder (0 exkluderade)

Sparade: C:\Users\michael.brostrom\Documents\GitHub\statistik-freshdesk-linear\raw\linear\linear_backfill_20250501_20260604.json
Filstorlek: 1376 KB


In [5]:
# Cell 5 — Verifiering
import pprint

dates = [n["createdAt"][:10] for n in filtered if n.get("createdAt")]
print(f"Antal noder:     {len(filtered)}")
print(f"Äldsta createdAt: {min(dates) if dates else 'N/A'}")
print(f"Senaste createdAt: {max(dates) if dates else 'N/A'}")

# Kontroll att nya fält finns
new_fields = ["completedAt", "canceledAt", "startedAt", "dueDate", "parent", "cycle"]
sample = filtered[0] if filtered else {}
missing = [f for f in new_fields if f not in sample]
print(f"\nNya fält närvarande: {[f for f in new_fields if f in sample]}")
if missing:
    print(f"SAKNAS: {missing}")

print("\nFörsta noden:")
pprint.pprint(sample)

Antal noder:     1089
Äldsta createdAt: 2025-09-12
Senaste createdAt: 2026-06-04

Nya fält närvarande: ['completedAt', 'canceledAt', 'startedAt', 'dueDate', 'parent', 'cycle']

Första noden:
{'archivedAt': None,
 'assignee': {'email': 'christoffer.thelenius@intersolia.com',
              'id': '475a06a0-4337-4f6e-b5d9-be826efc1bfa',
              'name': 'Christoffer Thelenius'},
 'canceledAt': None,
 'completedAt': None,
 'createdAt': '2026-06-04T06:36:53.712Z',
 'cycle': None,
 'description': 'Create a new database with all the specific CAT-data and '
                'migrate from iMaster/localRequested\n'
                '\n'
                '(Due to replication errors)',
 'dueDate': None,
 'estimate': None,
 'id': '63237d6b-bdc4-4e2b-a025-83f9af4e919c',
 'identifier': 'DEV-217',
 'labels': {'nodes': []},
 'number': 217,
 'parent': None,
 'priority': 0,
 'project': {'id': 'd9c9e1a0-2153-49e7-9c0f-b50a862d86c2',
             'name': 'Intersolia Customer Tool'},
 'startedAt': None,
 '